In [ ]:
# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from statsmodels.tsa.holtwinters import ExponentialSmoothing
from statsmodels.tsa.seasonal import seasonal_decompose

pd.set_option("display.max_columns", None)
plt.rcParams["figure.figsize"] = (12, 5)


In [ ]:
# Upload and load the dataset
from google.colab import files
import io

uploaded = files.upload()

file_name = next(iter(uploaded))
df = pd.read_csv(io.BytesIO(uploaded[file_name]), encoding="latin1")

print("File loaded:", file_name)
print("Dataset shape:", df.shape)
display(df.head())


In [ ]:
# Inspect the dataset
print("Columns:")
print(df.columns.tolist())

print("\nData types:")
print(df.dtypes)

print("\nMissing values:")
display(df.isnull().sum().sort_values(ascending=False).to_frame("Missing Values"))

print("\nDuplicate rows:", df.duplicated().sum())


In [ ]:
# Clean and prepare the data

# Standardize column names
df.columns = [str(col).strip() for col in df.columns]

# Convert date column
df["Date"] = pd.to_datetime(df["Date"], errors="coerce")

# Convert sales to numeric
df["Weekly_Sales"] = pd.to_numeric(df["Weekly_Sales"], errors="coerce")

# Remove invalid rows
df = df.dropna(subset=["Date", "Weekly_Sales"]).drop_duplicates().copy()

# Sort records
df = df.sort_values("Date")

print("Cleaned shape:", df.shape)
print("Date range:", df["Date"].min().date(), "to", df["Date"].max().date())

display(df.head())


In [ ]:
# Aggregate sales across all stores and departments by week

weekly_sales = (
    df.groupby("Date", as_index=True)["Weekly_Sales"]
      .sum()
      .sort_index()
)

# Make the time series frequency weekly
weekly_sales = weekly_sales.asfreq("W-FRI")

# Fill any missing weekly observations using time interpolation
weekly_sales = weekly_sales.interpolate(method="time")

print("Weekly observations:", len(weekly_sales))
display(weekly_sales.head(10).to_frame("Total Sales"))


In [ ]:
# Basic sales statistics

total_sales = weekly_sales.sum()
average_weekly_sales = weekly_sales.mean()
highest_week = weekly_sales.idxmax()
highest_week_sales = weekly_sales.max()
lowest_week = weekly_sales.idxmin()
lowest_week_sales = weekly_sales.min()

print(f"Total Sales: {total_sales:,.2f}")
print(f"Average Weekly Sales: {average_weekly_sales:,.2f}")
print(f"Highest Sales Week: {highest_week.date()} | {highest_week_sales:,.2f}")
print(f"Lowest Sales Week: {lowest_week.date()} | {lowest_week_sales:,.2f}")

# Weekly sales trend
plt.figure()
plt.plot(weekly_sales.index, weekly_sales.values)
plt.title("Historical Weekly Sales")
plt.xlabel("Date")
plt.ylabel("Sales")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


In [ ]:
# Analyze seasonality and peak demand periods

monthly_sales = weekly_sales.resample("ME").sum()

seasonal_month = monthly_sales.groupby(monthly_sales.index.month).mean()
seasonal_month.index = [
    "Jan", "Feb", "Mar", "Apr", "May", "Jun",
    "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"
]

print("Average sales by calendar month:")
display(seasonal_month.sort_values(ascending=False).to_frame("Average Sales"))

plt.figure()
plt.plot(seasonal_month.index, seasonal_month.values, marker="o")
plt.title("Average Sales by Month")
plt.xlabel("Month")
plt.ylabel("Average Sales")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

# Top peak weeks
peak_weeks = weekly_sales.sort_values(ascending=False).head(10)
print("Top 10 peak sales weeks:")
display(peak_weeks.to_frame("Weekly Sales"))


In [ ]:
# Seasonal decomposition
# The dataset contains roughly two years of weekly observations,
# so a 52-week seasonal cycle is used.

decomposition = seasonal_decompose(
    weekly_sales,
    model="additive",
    period=52
)

decomposition.plot()
plt.tight_layout()
plt.show()


In [ ]:
# Train/test split for model evaluation

# Keep the last 12 weeks as a test set
test_size = 12

train = weekly_sales.iloc[:-test_size]
test = weekly_sales.iloc[-test_size:]

print("Training observations:", len(train))
print("Testing observations:", len(test))
print("Training period:", train.index.min().date(), "to", train.index.max().date())
print("Testing period:", test.index.min().date(), "to", test.index.max().date())


In [ ]:
# Basic time-series forecasting model
# Holt-Winters Exponential Smoothing with weekly seasonality

model = ExponentialSmoothing(
    train,
    trend="add",
    seasonal="add",
    seasonal_periods=52,
    initialization_method="estimated"
)

fitted_model = model.fit(optimized=True)

test_forecast = fitted_model.forecast(test_size)

# Evaluation metrics
mae = np.mean(np.abs(test.values - test_forecast.values))
rmse = np.sqrt(np.mean((test.values - test_forecast.values) ** 2))

# Avoid division by zero in MAPE
non_zero_actual = test.values != 0
mape = (
    np.mean(
        np.abs(
            (test.values[non_zero_actual] - test_forecast.values[non_zero_actual])
            / test.values[non_zero_actual]
        )
    ) * 100
)

print(f"MAE: {mae:,.2f}")
print(f"RMSE: {rmse:,.2f}")
print(f"MAPE: {mape:.2f}%")

plt.figure()
plt.plot(train.index, train.values, label="Training Sales")
plt.plot(test.index, test.values, label="Actual Sales")
plt.plot(test.index, test_forecast.values, label="Predicted Sales")
plt.title("Actual vs Predicted Sales")
plt.xlabel("Date")
plt.ylabel("Sales")
plt.legend()
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


In [ ]:
# Refit model on the complete historical data and forecast future sales

forecast_horizon = 12

final_model = ExponentialSmoothing(
    weekly_sales,
    trend="add",
    seasonal="add",
    seasonal_periods=52,
    initialization_method="estimated"
)

final_fitted_model = final_model.fit(optimized=True)

future_forecast = final_fitted_model.forecast(forecast_horizon)

forecast_df = future_forecast.to_frame(name="Forecasted_Sales")
forecast_df["Lower_Band"] = forecast_df["Forecasted_Sales"] * 0.90
forecast_df["Upper_Band"] = forecast_df["Forecasted_Sales"] * 1.10

display(forecast_df)


In [ ]:
# Visualize historical sales and future forecast

plt.figure(figsize=(14, 6))

plt.plot(
    weekly_sales.index,
    weekly_sales.values,
    label="Historical Sales"
)

plt.plot(
    future_forecast.index,
    future_forecast.values,
    marker="o",
    label="Forecast"
)

plt.fill_between(
    forecast_df.index,
    forecast_df["Lower_Band"],
    forecast_df["Upper_Band"],
    alpha=0.2,
    label="Approx. Forecast Range"
)

plt.axvline(
    weekly_sales.index[-1],
    linestyle="--",
    label="Forecast Start"
)

plt.title("Historical and Forecasted Retail Sales")
plt.xlabel("Date")
plt.ylabel("Weekly Sales")
plt.legend()
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


In [ ]:
# Store-level analysis for inventory planning

store_summary = (
    df.groupby("Store")
      .agg(
          Total_Sales=("Weekly_Sales", "sum"),
          Average_Weekly_Sales=("Weekly_Sales", "mean"),
          Peak_Weekly_Sales=("Weekly_Sales", "max")
      )
      .sort_values("Total_Sales", ascending=False)
)

print("Top 10 stores by total sales:")
display(store_summary.head(10))


In [ ]:
# Inventory and restocking recommendations

forecast_average = future_forecast.mean()
forecast_max = future_forecast.max()
forecast_min = future_forecast.min()

recent_average = weekly_sales.tail(12).mean()
recent_max = weekly_sales.tail(12).max()

growth_vs_recent = ((forecast_average - recent_average) / recent_average) * 100

print("INVENTORY / RESTOCKING RECOMMENDATIONS")
print("-" * 50)
print(f"Average forecasted weekly sales: {forecast_average:,.2f}")
print(f"Maximum forecasted weekly sales: {forecast_max:,.2f}")
print(f"Minimum forecasted weekly sales: {forecast_min:,.2f}")
print(f"Recent 12-week average sales: {recent_average:,.2f}")
print(f"Forecast vs recent average: {growth_vs_recent:.2f}%")

if growth_vs_recent > 5:
    print("\nRecommendation: Increase inventory levels and place replenishment orders earlier.")
elif growth_vs_recent < -5:
    print("\nRecommendation: Reduce excess inventory and avoid aggressive restocking.")
else:
    print("\nRecommendation: Maintain normal inventory levels and monitor weekly demand.")

if forecast_max > recent_max:
    print("Peak-demand alert: Future forecast exceeds the recent 12-week peak.")
    print("Keep additional safety stock available for high-demand periods.")
else:
    print("No forecasted week exceeds the recent 12-week peak.")


In [ ]:
# Final summary

peak_forecast_date = future_forecast.idxmax()
peak_forecast_value = future_forecast.max()

print("FINAL PROJECT SUMMARY")
print("=" * 50)
print(f"Historical period: {weekly_sales.index.min().date()} to {weekly_sales.index.max().date()}")
print(f"Historical total sales: {total_sales:,.2f}")
print(f"Model MAPE on 12-week test period: {mape:.2f}%")
print(f"Forecast horizon: {forecast_horizon} weeks")
print(f"Highest forecasted week: {peak_forecast_date.date()}")
print(f"Highest forecasted sales: {peak_forecast_value:,.2f}")
print(f"Forecast average weekly sales: {forecast_average:,.2f}")

print("\nRecommended action:")
if growth_vs_recent > 5:
    print("Prepare for higher demand by increasing safety stock and replenishment frequency.")
elif growth_vs_recent < -5:
    print("Demand is expected to soften; control inventory and reduce unnecessary stock.")
else:
    print("Demand is relatively stable; maintain normal stock levels and monitor the forecast.")
